# 📊 Notebook 4 — Model Evaluation & Comparison

**Mục tiêu notebook:**
- **Bước 12:** So sánh toàn diện 3 models: Accuracy, Precision, Recall, F1, ROC-AUC
- **Bước 13:** Kết luận model tốt nhất + Feature Importance + Error Analysis
- **Bước 14:** Threshold Optimization — tìm ngưỡng tối ưu theo F1/Business cost
- **Bước 15:** KPI Check — Business Impact Simulation (chi phí lỗi phân loại)

---

## 🗺️ Lộ trình 15 bước — Pipeline ML đầy đủ

| # | Bước | Notebook |
|:-:|------|----------|
| 1 | Đặt vấn đề | NB1 |
| 2 | Thu thập dữ liệu | NB1 |
| 3 | Tiền xử lý & EDA | NB2 |
| 4 | Định nghĩa Target | NB2 |
| 5 | Feature Engineering | NB3 |
| 6 | Feature Selection | NB3 |
| 7 | Train/Test Split + Scale | NB3 |
| 8 | Chọn mô hình | NB3 |
| 9 | Huấn luyện | NB3 |
| 10 | Đánh giá baseline | NB3 |
| 11 | Feature Reduction | NB3 |
| **12** | **So sánh & Phân tích sâu** | **📌 NB4** |
| **13** | **Kết luận Model tốt nhất** | **📌 NB4** |
| **14** | **Threshold Optimization** | **📌 NB4** |
| **15** | **KPI Check** | **📌 NB4** |

> 💡 **Notebook này bao phủ: Bước 12 → 15 / 15** — Đánh giá, so sánh và đưa ra quyết định cuối cùng về model deployment

In [ ]:
# ============================================================
# SECTION 0: Import thư viện và Load models
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import joblib
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, precision_recall_curve,
    confusion_matrix, classification_report
)

# --- Thiết lập ---
RANDOM_STATE = 42
pd.set_option('display.float_format', '{:,.4f}'.format)
plt.rcParams.update({
    'figure.dpi'       : 120,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'        : 11
})
PALETTE_3 = ['#E74C3C', '#F39C12', '#2ECC71']   # LR, XGBoost, LGBM

# --- Đường dẫn ---
BASE_DIR   = Path('.').resolve()
DATA_DIR   = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'

print('✅ Libraries loaded')

In [ ]:

# ============================================================
# Load tất cả models, 3 optimal thresholds và test/val data
# ============================================================
lr_model   = joblib.load(MODELS_DIR / 'lr_model.pkl')
xgb_model  = joblib.load(MODELS_DIR / 'xgb_model.pkl')
lgbm_model = joblib.load(MODELS_DIR / 'best_model.pkl')

# 3 thresholds riêng biệt — mỗi model có threshold tối ưu từ Validation set
lr_opt_thresh   = joblib.load(MODELS_DIR / 'lr_optimal_threshold.pkl')
xgb_opt_thresh  = joblib.load(MODELS_DIR / 'xgb_optimal_threshold.pkl')
lgbm_opt_thresh = joblib.load(MODELS_DIR / 'optimal_threshold.pkl')

test_data = joblib.load(MODELS_DIR / 'test_data.pkl')

# Test data — dùng để đánh giá cuối cùng
X_test    = test_data['X_test']      # unscaled — XGBoost / LightGBM
X_test_lr = test_data['X_test_lr']  # scaled   — Logistic Regression
y_test    = test_data['y_test']

# Val data — lưu kèm (không dùng trực tiếp trong NB4; threshold đã tính sẵn)
X_val    = test_data['X_val']
y_val    = test_data['y_val']

print('✅ Loaded models: LR | XGBoost | LightGBM')
print()
print('   Optimal Thresholds (tuned trên Validation set):')
print(f'   LR       → lr_opt_thresh   = {lr_opt_thresh:.4f}')
print(f'   XGBoost  → xgb_opt_thresh  = {xgb_opt_thresh:.4f}')
print(f'   LightGBM → lgbm_opt_thresh = {lgbm_opt_thresh:.4f}')
print()
print(f'   Test set (unscaled) : {X_test.shape[0]:,} dòng × {X_test.shape[1]} features  → XGBoost / LightGBM')
print(f'   Test set (LR scaled): {X_test_lr.shape[0]:,} dòng × {X_test_lr.shape[1]} features  → Logistic Regression')
print(f'   Val  set            : {X_val.shape[0]:,} dòng × {X_val.shape[1]} features  (saved for reference)')
print()
print('   ℹ️  Features: 29 (14 raw + 15 derived)  — dominant_gender/location đã loại')
print('   ℹ️  Pipeline: 70% Train / 15% Val / 15% Test (3-way split — chuẩn công nghiệp)')


In [ ]:

# ============================================================
# Tính probabilities và metrics cho tất cả models
# Threshold: mỗi model dùng threshold tối ưu riêng (từ Validation set)
# ============================================================
import time

results_dict = {}
PALETTE_3 = ['#E74C3C', '#F39C12', '#2ECC71']   # LR, XGBoost, LGBM

# Model configs: (model, X_test_dùng, threshold_tối_ưu, tên)
# LR      → X_test_lr  (scaled  — StandardScaler)
# XGBoost → X_test     (unscaled — tree model)
# LightGBM→ X_test     (unscaled — tree model)
model_configs = [
    (lr_model,   X_test_lr, lr_opt_thresh,   'Logistic Regression'),
    (xgb_model,  X_test,    xgb_opt_thresh,  'XGBoost'),
    (lgbm_model, X_test,    lgbm_opt_thresh, 'LightGBM'),
]

probs_dict = {}
print(f'  {"Model":<22} {"F1":>8} {"AUC":>8} {"Prec":>8} {"Rec":>8} {"θ (Val)":>10}')
print(f'  {"─"*65}')
for model, X, thresh, name in model_configs:
    t0 = time.time()
    probs  = model.predict_proba(X)[:, 1]
    preds  = (probs >= thresh).astype(int)
    inf_ms = (time.time() - t0) * 1000

    probs_dict[name] = probs
    results_dict[name] = {
        'Accuracy'      : accuracy_score(y_test, preds),
        'Precision'     : precision_score(y_test, preds, zero_division=0),
        'Recall'        : recall_score(y_test, preds, zero_division=0),
        'F1-Score'      : f1_score(y_test, preds, zero_division=0),
        'ROC-AUC'       : roc_auc_score(y_test, probs),
        'Threshold'     : thresh,
        'Inference (ms)': inf_ms,
    }
    res = results_dict[name]
    print(f'  {name:<22} {res["F1-Score"]:>8.4f} {res["ROC-AUC"]:>8.4f} '
          f'{res["Precision"]:>8.4f} {res["Recall"]:>8.4f} {thresh:>10.4f}')

print(f'\n  ✅ Predictions computed for all 3 models')
print(f'  ✅ Mỗi model dùng threshold riêng — tìm trên Validation set (fair comparison)')



---
## 📌 Bước 12 / 15 — So sánh & Phân tích sâu

### 1. Bảng So Sánh Tổng Hợp


In [ ]:
# ============================================================
# SECTION 1: Bảng so sánh — Styled DataFrame
# ============================================================
results_df = pd.DataFrame(results_dict).T
results_df = results_df.astype(float).round(4)

# Highlight: xanh = best metric mỗi cột, đỏ = worst
metric_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']

def highlight_best(s):
    """Tô màu xanh cho giá trị cao nhất, đỏ cho thấp nhất."""
    if s.name in metric_cols:
        is_best  = s == s.max()
        is_worst = s == s.min()
        colors = []
        for b, w in zip(is_best, is_worst):
            if b:
                colors.append('background-color: #D5F5E3; color: #1E8449; font-weight: bold')
            elif w:
                colors.append('background-color: #FADBD8; color: #C0392B')
            else:
                colors.append('')
        return colors
    return [''] * len(s)

styled = results_df.style.apply(highlight_best, axis=0)
print('📊 BẢNG SO SÁNH 3 MODELS (🟢 = best metric | 🔴 = worst):')
styled

## 2. ROC Curve — 3 Models

In [ ]:
# ============================================================
# SECTION 2: ROC Curve 3 models trên cùng 1 plot
# ============================================================
fig, ax = plt.subplots(figsize=(9, 6))

for (name, probs), color in zip(probs_dict.items(), PALETTE_3):
    fpr, tpr, _ = roc_curve(y_test, probs)
    auc = roc_auc_score(y_test, probs)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name}  (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Random Classifier')
ax.fill_between([0, 1], [0, 1], alpha=0.05, color='gray')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve — 3 Models Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 3. Precision-Recall Curve — 3 Models

In [ ]:
# ============================================================
# SECTION 3: Precision-Recall Curve 3 models
# ============================================================
fig, ax = plt.subplots(figsize=(9, 6))

for (name, probs), color in zip(probs_dict.items(), PALETTE_3):
    precision, recall, _ = precision_recall_curve(y_test, probs)
    # Average Precision
    from sklearn.metrics import average_precision_score
    ap = average_precision_score(y_test, probs)
    ax.plot(recall, precision, color=color, linewidth=2.5,
            label=f'{name}  (AP = {ap:.4f})')

# Baseline: random classifier
baseline = y_test.mean()
ax.axhline(baseline, color='gray', linestyle='--', linewidth=1,
           label=f'No-Skill ({baseline:.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve — 3 Models Comparison', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

## 4. Confusion Matrix — 3 Models (Side-by-Side)

In [ ]:

# ============================================================
# SECTION 4: Confusion Matrix 1×3 subplots
# Mỗi model dùng threshold tối ưu riêng (tìm trên Validation set)
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmaps = ['Reds', 'Oranges', 'Greens']
thresholds_cm = [lr_opt_thresh, xgb_opt_thresh, lgbm_opt_thresh]

for (name, probs), color, thresh, ax, cmap in zip(
    probs_dict.items(), PALETTE_3,
    thresholds_cm,
    axes, cmaps
):
    preds = (probs >= thresh).astype(int)
    cm    = confusion_matrix(y_test, preds)

    f1  = f1_score(y_test, preds, zero_division=0)
    auc = roc_auc_score(y_test, probs)

    sns.heatmap(
        cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=['Pred 0', 'Pred 1'],
        yticklabels=['Actual 0', 'Actual 1'],
        ax=ax, linewidths=0.5, linecolor='white',
        annot_kws={'size': 14, 'fontweight': 'bold'}
    )
    ax.set_title(f'{name}\nθ_val={thresh:.3f} | F1={f1:.4f} | AUC={auc:.4f}',
                 fontsize=11, fontweight='bold')

plt.suptitle('Confusion Matrix — 3 Models (θ tối ưu từ Validation set)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 5. Feature Importance: LightGBM vs XGBoost

In [ ]:
# ============================================================
# SECTION 5: Feature Importance — Grouped bar chart (top 15)
# LightGBM vs XGBoost
# ============================================================
fi_df = pd.read_csv(MODELS_DIR / 'feature_importance.csv')

# Normalize để so sánh cùng scale
fi_df['LGBM_norm'] = fi_df['LightGBM_Importance'] / fi_df['LightGBM_Importance'].sum()
fi_df['XGB_norm']  = fi_df['XGBoost_Importance']  / fi_df['XGBoost_Importance'].sum()

# Top 15 theo LightGBM
top15 = fi_df.sort_values('LGBM_norm', ascending=False).head(15)

x  = np.arange(len(top15))
w  = 0.35

fig, ax = plt.subplots(figsize=(14, 7))
bars1 = ax.barh(x + w/2, top15['LGBM_norm'].values[::-1] * 100,
                height=w, color='#2ECC71', edgecolor='white', label='LightGBM')
bars2 = ax.barh(x - w/2, top15['XGB_norm'].values[::-1] * 100,
                height=w, color='#F39C12', edgecolor='white', label='XGBoost')

ax.set_yticks(x)
ax.set_yticklabels(top15['Feature'].values[::-1])
ax.set_xlabel('Normalized Importance (%)')
ax.set_title('Feature Importance: LightGBM vs XGBoost (Top 15)', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

print('\n📋 TOP 10 FEATURES theo LightGBM:')
print(top15[['Feature', 'LGBM_norm', 'XGB_norm']].head(10).to_string(index=False))



---
## 📌 Bước 13 / 15 — Kết luận Model Tốt Nhất

### 7. Kết luận & Đề xuất


In [ ]:

# ============================================================
# SECTION 7A: Radar Chart — So sánh 5 metrics, 3 models
# ============================================================
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
models  = list(results_dict.keys())
N       = len(metrics)
angles  = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig = plt.figure(figsize=(14, 6))
gs  = fig.add_gridspec(1, 2, width_ratios=[1, 1.2])

# ── Subplot trái: Radar chart
ax_radar = fig.add_subplot(gs[0], projection='polar')
for name, color in zip(models, PALETTE_3):
    vals = [results_dict[name][m] for m in metrics]
    vals += vals[:1]
    ax_radar.plot(angles, vals, 'o-', linewidth=2.5, color=color, label=name)
    ax_radar.fill(angles, vals, alpha=0.08, color=color)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metrics, fontsize=10)
ax_radar.set_ylim(0.5, 1.0)
ax_radar.set_yticks([0.6, 0.7, 0.8, 0.9])
ax_radar.set_yticklabels(['0.6', '0.7', '0.8', '0.9'], fontsize=8)
ax_radar.set_title('Radar: 5 Metrics × 3 Models', fontsize=12, fontweight='bold', pad=15)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.4, 1.15), fontsize=10)

# ── Subplot phải: Score Distribution (class separation)
ax_dist = fig.add_subplot(gs[1])
offsets = {'Logistic Regression': -0.25, 'XGBoost': 0.0, 'LightGBM': 0.25}

for i, (name, color) in enumerate(zip(models, PALETTE_3)):
    probs_0 = probs_dict[name][y_test.values == 0]
    probs_1 = probs_dict[name][y_test.values == 1]
    x_off = i * 0.3
    vp0 = ax_dist.violinplot([probs_0], positions=[x_off], widths=0.25, showmedians=True)
    vp1 = ax_dist.violinplot([probs_1], positions=[x_off + 0.12], widths=0.25, showmedians=True)
    for body in vp0['bodies']: body.set_facecolor('#3498DB'); body.set_alpha(0.5)
    for body in vp1['bodies']: body.set_facecolor('#E74C3C'); body.set_alpha(0.5)
    for part in ['cbars', 'cmins', 'cmaxes', 'cmedians']:
        if part in vp0: vp0[part].set_color('#3498DB')
        if part in vp1: vp1[part].set_color('#E74C3C')

ax_dist.set_xticks([0.06, 0.36, 0.66])
ax_dist.set_xticklabels(['Logistic\nRegression', 'XGBoost', 'LightGBM'], fontsize=10)
ax_dist.set_ylabel('Predicted Probability')
ax_dist.set_title('Score Distribution\n🔵 Class 0 (không tái mua)  🔴 Class 1 (tái mua)', fontsize=11)
ax_dist.axhline(0.5, color='gray', linestyle='--', alpha=0.6, linewidth=1)

fig.suptitle('Section 7: Radar Comparison & Score Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── Kết luận Best Model ──────────────────────────────────────────────────────
best_name = max(results_dict, key=lambda n: results_dict[n]['F1-Score'])
best_f1   = results_dict[best_name]['F1-Score']
best_auc  = results_dict[best_name]['ROC-AUC']

# Lý do lựa chọn — động theo tên model thực tế win
model_reasons = {
    'LightGBM': [
        '→ Leaf-wise tree growth → hội tụ nhanh trên dữ liệu lớn',
        '→ class_weight=balanced xử lý tốt imbalance (label 0/1 lệch)',
        '→ Early Stopping trên Val set — chống overfitting',
        '→ F1-Score và ROC-AUC cao nhất trong 3 models',
    ],
    'XGBoost': [
        '→ Level-wise boosting mạnh mẽ, regularization (L1/L2) tích hợp',
        '→ scale_pos_weight xử lý tốt class imbalance',
        '→ Early Stopping trên Val set — chống overfitting',
        '→ F1-Score và ROC-AUC cao nhất trong 3 models',
    ],
    'Logistic Regression': [
        '→ Linear baseline interpretable — dễ giải thích cho stakeholder',
        '→ class_weight=balanced xử lý class imbalance',
        '→ Threshold tuning trên Val set cho F1 tối ưu',
        '→ F1-Score cao nhất trong 3 models',
    ],
}

print('\n' + '='*68)
print('  🏆 KẾT LUẬN — MODEL TỐT NHẤT (Bước 13)')
print('='*68)
print(f'  {"Model":<25} {"F1-Score":>9} {"ROC-AUC":>9} {"Recall":>9} {"Precision":>10} {"θ*":>8}')
print(f'  {"-"*72}')
for name, res in results_dict.items():
    tag = '  ← ✅ BEST' if name == best_name else ''
    print(f'  {name:<25} {res["F1-Score"]:>9.4f} {res["ROC-AUC"]:>9.4f} '
          f'{res["Recall"]:>9.4f} {res["Precision"]:>10.4f} {res["Threshold"]:>8.4f}{tag}')
print(f'  {"-"*72}')
print(f'\n  ✅ Model được chọn: {best_name}')
print(f'     F1-Score = {best_f1:.4f}  |  ROC-AUC = {best_auc:.4f}')
print(f'\n  📋 Lý do lựa chọn:')
for reason in model_reasons.get(best_name, ['→ F1-Score cao nhất trong 3 models']):
    print(f'     {reason}')
print(f'\n  ℹ️  Threshold tối ưu θ* tìm trên Validation set → không rò rỉ thông tin Test')
print(f'     Model này được dùng ở Bước 14 (Threshold) & Bước 15 (KPI)')
print('='*68)



---
## 📌 Bước 14 / 15 — Threshold Optimization

> **Mục tiêu:** Tìm ngưỡng phân loại tối ưu θ\* để cân bằng Precision / Recall / F1 theo từng model.  
> Threshold mặc định = 0.5 không phải luôn tốt nhất — đặc biệt với dữ liệu imbalance.

### 8. Threshold Sensitivity — F1 vs Threshold (3 Models)


In [ ]:

# ============================================================
# SECTION 8: Threshold Sensitivity — F1 vs Threshold cho 3 models
#
# Đường cong F1/Precision/Recall tính trên TEST set (hiển thị)
# Chấm đen = θ* tối ưu tìm trên VALIDATION set (không rò rỉ)
# ============================================================
thresholds_range = np.linspace(0.10, 0.90, 60)

# Threshold tối ưu từ Validation set — mỗi model có threshold riêng
opt_thresholds = {
    'Logistic Regression': lr_opt_thresh,
    'XGBoost'            : xgb_opt_thresh,
    'LightGBM'           : lgbm_opt_thresh,
}

fig, axes = plt.subplots(1, 3, figsize=(17, 5), sharey=True)

for (name, probs), color, ax in zip(probs_dict.items(), PALETTE_3, axes):
    f1_vals   = [f1_score(y_test, (probs >= t).astype(int), zero_division=0) for t in thresholds_range]
    prec_vals = [precision_score(y_test, (probs >= t).astype(int), zero_division=0) for t in thresholds_range]
    rec_vals  = [recall_score(y_test, (probs >= t).astype(int), zero_division=0) for t in thresholds_range]

    ax.plot(thresholds_range, f1_vals,   color=color,    linewidth=2.5, label='F1-Score (Test)')
    ax.plot(thresholds_range, prec_vals, color='#2980B9', linewidth=1.5, linestyle='--', label='Precision (Test)')
    ax.plot(thresholds_range, rec_vals,  color='#E74C3C', linewidth=1.5, linestyle=':',  label='Recall (Test)')

    opt_t   = opt_thresholds[name]
    best_f1 = f1_score(y_test, (probs >= opt_t).astype(int), zero_division=0)
    ax.axvline(opt_t, color='black', linestyle='--', linewidth=1.5,
               label=f'θ_val={opt_t:.3f}  F1_test={best_f1:.3f}')
    ax.scatter([opt_t], [best_f1], color='black', s=80, zorder=5)

    ax.set_xlabel('Threshold (ngưỡng phân loại)')
    ax.set_title(f'{name}', fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlim(0.1, 0.9)
    ax.set_ylim(0, 1.0)

axes[0].set_ylabel('Score')
fig.suptitle(
    '📈 Threshold Sensitivity: F1 / Precision / Recall vs Threshold\n'
    '(Đường cong: Test set | Chấm đen: θ* tối ưu từ Validation set)',
    fontsize=12, fontweight='bold'
)
plt.tight_layout()
plt.show()

print('   θ* được tìm trên Validation set → apply lên Test set để đánh giá')
print(f'   LR       θ* = {lr_opt_thresh:.4f}   F1_test = {f1_score(y_test, (probs_dict["Logistic Regression"] >= lr_opt_thresh).astype(int), zero_division=0):.4f}')
print(f'   XGBoost  θ* = {xgb_opt_thresh:.4f}   F1_test = {f1_score(y_test, (probs_dict["XGBoost"] >= xgb_opt_thresh).astype(int), zero_division=0):.4f}')
print(f'   LightGBM θ* = {lgbm_opt_thresh:.4f}   F1_test = {f1_score(y_test, (probs_dict["LightGBM"] >= lgbm_opt_thresh).astype(int), zero_division=0):.4f}')



---
## 📌 Bước 15 / 15 — KPI Check (Business Impact Simulation)

> **Giả định chi phí kinh doanh:**
> - **FN** (bỏ lỡ khách tái mua thực sự) = mất cơ hội upsell → **−150,000 VNĐ/người**
> - **FP** (gửi voucher nhầm cho khách không tái mua) = lãng phí voucher → **−30,000 VNĐ/người**
>
> Model tốt về F1 nhưng nếu FN cost >> FP cost → cần tăng Recall, giảm Threshold.

### 9. Business Impact Simulation — Chi Phí Lỗi Phân Loại


In [ ]:

# ============================================================
# SECTION 9: Business Impact Simulation
# Giả định chi phí kinh doanh:
#   FN (False Negative): bỏ lỡ khách tái mua = mất cơ hội upsell → -150,000 VND/người
#   FP (False Positive): gửi voucher nhầm     = lãng phí voucher  →  -30,000 VND/người
# ============================================================
REVENUE_PER_REPURCHASER = 150_000
VOUCHER_COST            =  30_000

thresholds_biz = np.linspace(0.10, 0.90, 80)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Plot 1: Tổng chi phí lỗi theo threshold
for (name, probs), color in zip(probs_dict.items(), PALETTE_3):
    losses = []
    for t in thresholds_biz:
        preds = (probs >= t).astype(int)
        fn = int(((y_test.values == 1) & (preds == 0)).sum())
        fp = int(((y_test.values == 0) & (preds == 1)).sum())
        losses.append((fn * REVENUE_PER_REPURCHASER + fp * VOUCHER_COST) / 1e6)
    axes[0].plot(thresholds_biz, losses, color=color, linewidth=2.5, label=name)

axes[0].set_xlabel('Threshold')
axes[0].set_ylabel('Tổng Chi Phí Lỗi (Triệu VND)')
axes[0].set_title('Chi Phí Lỗi Phân Loại vs Threshold\n'
                  f'(FN={REVENUE_PER_REPURCHASER:,} VND/người | FP={VOUCHER_COST:,} VND/người)',
                  fontsize=11, fontweight='bold')
axes[0].legend()

# ── Plot 2: Tại threshold tối ưu riêng mỗi model — so sánh chi phí lỗi
opt_thrs = {
    'Logistic Regression': lr_opt_thresh,
    'XGBoost'            : xgb_opt_thresh,
    'LightGBM'           : lgbm_opt_thresh,
}
model_names_list, fn_costs, fp_costs = [], [], []

for name, probs in probs_dict.items():
    t    = opt_thrs[name]
    preds = (probs >= t).astype(int)
    fn   = int(((y_test.values == 1) & (preds == 0)).sum())
    fp   = int(((y_test.values == 0) & (preds == 1)).sum())
    model_names_list.append(name)
    fn_costs.append(fn * REVENUE_PER_REPURCHASER / 1e6)
    fp_costs.append(fp * VOUCHER_COST / 1e6)

x   = np.arange(len(model_names_list))
w   = 0.35
b1  = axes[1].bar(x - w/2, fn_costs, w, label='FN Cost (mất doanh thu)', color='#E74C3C', edgecolor='white')
b2  = axes[1].bar(x + w/2, fp_costs, w, label='FP Cost (voucher lãng phí)', color='#F39C12', edgecolor='white')

for bar in b1:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.1f}M', ha='center', fontsize=9)
for bar in b2:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f'{bar.get_height():.1f}M', ha='center', fontsize=9)

axes[1].set_xticks(x)
axes[1].set_xticklabels(model_names_list, fontsize=10)
axes[1].set_ylabel('Chi Phí (Triệu VND)')
axes[1].set_title('So Sánh Chi Phí Lỗi tại θ* Tối Ưu\n(3 Models, Test Set — θ* tìm trên Validation set)',
                  fontsize=11, fontweight='bold')
axes[1].legend()

fig.suptitle(' Business Impact: Chi Phí Phân Loại Sai theo Kịch Bản Kinh Doanh',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('   Chi tiết threshold và chi phí lỗi:')
print(f'   {"Model":<22} {"θ* (Val)":>10} {"FN Cost (M)":>13} {"FP Cost (M)":>13} {"Total (M)":>11}')
print(f'   {"─"*72}')
for name, fn_c, fp_c in zip(model_names_list, fn_costs, fp_costs):
    print(f'   {name:<22} {opt_thrs[name]:>10.4f} {fn_c:>13.2f} {fp_c:>13.2f} {fn_c + fp_c:>11.2f}')



---
## 📌 Đề Xuất Hướng Giải Quyết — 2 Kịch Bản Chiến Lược

> **Bối cảnh phân tích:** Dựa trên kết quả mô phỏng chi phí ở Bước 15, doanh nghiệp đối mặt với bài toán đánh đổi cốt lõi: **FN Cost** (mất doanh thu từ khách tái mua bị bỏ sót, −150,000 VND/người) vs **FP Cost** (lãng phí ngân sách voucher cho khách không có nhu cầu, −30,000 VND/người). Vì **FN đắt gấp 5× FP**, chiến lược tối ưu sẽ thay đổi tùy theo **giai đoạn kinh doanh** và **mục tiêu thị trường** của doanh nghiệp.

---

### 🟢 Kịch Bản 1 — Tối Ưu Hóa Chi Phí Dài Hạn · Vận Hành Bền Vững

**Model đề xuất:** LightGBM tại ngưỡng **θ = 0.5041** (threshold tối ưu F1 từ Validation set)

| Chỉ số tài chính | Giá trị | Nhận xét |
|------------------|---------|----------|
| FN Cost — Mất Doanh Thu | **45.0M VND** | Rủi ro bỏ sót được kiểm soát ở mức chấp nhận được |
| FP Cost — Voucher Lãng Phí | **55.9M VND** | Ngân sách Marketing phân bổ có chủ đích, đúng tệp |
| **Tổng Thiệt Hại** | **100.9M VND** | ✅ Mức thiệt hại tổng thể **thấp nhất** trong tất cả cấu hình |

**Lý do chiến lược:**
- **An toàn & Cân đối ngân sách:** Tổng thiệt hại 100.9M VND là mức **kiểm soát được nhất**, phù hợp với chu kỳ vận hành thường ngày khi Marketing không có room đặc biệt để "đầu tư thêm".
- **Precision cao hơn — đúng người, đúng ưu đãi:** LightGBM ở θ=0.5041 chỉ kích hoạt voucher cho tệp khách hàng có xác suất tái mua **đủ cao**, giảm thiểu chi phí phân phối voucher nhầm đối tượng.
- **Tốc độ inference cực nhanh:** LightGBM xử lý hàng chục nghìn records trong milliseconds — lý tưởng cho **hệ thống production real-time** với traffic cao, đặc biệt trong các giờ peak.
- **Phù hợp với:** Vận hành hằng ngày, giai đoạn kinh doanh bình thường, khi ngân sách Marketing cần được quản lý chặt chẽ và ROI phải được đảm bảo.

---

### 🟠 Kịch Bản 2 — Chiến Dịch Kích Cầu · Tăng Trưởng Thị Phần Ngắn Hạn

**Model đề xuất:** XGBoost tại ngưỡng thấp **θ = 0.3892** (giảm threshold để ép Recall cao hơn)

| Chỉ số tài chính | Giá trị | Nhận xét |
|------------------|---------|----------|
| FN Cost — Mất Doanh Thu | **29.7M VND** | ✅ Giảm **−15.3M** so với KB1 — bỏ sót ít khách hơn |
| FP Cost — Voucher Lãng Phí | **64.8M VND** | ⚠️ Tăng **+8.9M** — chấp nhận đánh đổi để phủ rộng |
| **Tổng Thiệt Hại** | **94.5M VND** | Tổng thấp hơn KB1 khi **FN cost >> FP cost** theo tỉ lệ 5:1 |

**Tư duy đánh đổi chiến lược:**
- **"Thà tốn thêm voucher còn hơn bỏ sót doanh thu":** Trong các **mùa cao điểm** (Tết Nguyên Đán, 11.11, Black Friday), **giá trị vòng đời khách hàng (CLV)** tăng đột biến — bỏ sót 1 khách tái mua tiêu tốn gấp 5× so với chi thêm 1 voucher.
- **Ngưỡng thấp = Recall cao = Phủ rộng tệp tiềm năng:** XGBoost ở θ=0.3892 kích hoạt voucher sớm hơn, đảm bảo **gần như không bỏ sót khách hàng có nhu cầu thực sự** — FN Cost bị nén xuống mức thấp kỷ lục 29.7M.
- **Linh hoạt hot-swap theo lịch campaign:** Pipeline deployment cho phép team Data **chuyển đổi model** giữa KB1 và KB2 mà không cần retrain — chỉ thay file `.pkl` và cập nhật threshold config.
- **Phù hợp với:** Flash sale, chiến dịch Upsell & Cross-sell mùa lễ, giai đoạn mở rộng thị phần khi doanh nghiệp ưu tiên **giữ chân và kích hoạt lại** tệp khách hàng tiềm năng.

---

> **💼 Khuyến nghị thực thi:** Duy trì **Kịch bản 1** (LightGBM θ=0.5041) làm **baseline vận hành hằng ngày**. Kích hoạt **Kịch bản 2** (XGBoost θ=0.3892) trong vòng **±7 ngày quanh các mùa lễ lớn**. Đánh giá hiệu quả theo tuần và điều chỉnh ngưỡng dựa trên phản hồi thực tế từ hệ thống production.


In [ ]:

# ============================================================
# SECTION 10: So Sanh Danh Doi 2 Kich Ban — Grouped Bar Chart
# KB1: LightGBM  theta=0.5041  (Ben Vung)
# KB2: XGBoost   theta=0.3892  (Tang Truong)
# ============================================================

CLR_KB1 = '#56B4E9'   # xanh da troi (sky blue)
CLR_KB2 = '#1565C0'   # xanh nuoc dai duong (ocean blue)

labels   = ['Mat Doanh Thu\n(FN Cost)', 'Lang Phi Voucher\n(FP Cost)']
kb1_vals = [45.0, 55.9]   # KB1: LightGBM theta = 0.5041
kb2_vals = [29.7, 64.8]   # KB2: XGBoost  theta = 0.3892

x = np.arange(len(labels))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 7.5))
fig.patch.set_facecolor('#F4F9FF')
ax.set_facecolor('#F4F9FF')

# --- Cot doi ---
bars1 = ax.bar(x - w/2, kb1_vals, w,
               color=CLR_KB1, alpha=0.87,
               label='Kich ban 1 - LightGBM (theta=0.5041)   Ben Vung',
               edgecolor='white', linewidth=1.5, zorder=3)
bars2 = ax.bar(x + w/2, kb2_vals, w,
               color=CLR_KB2, alpha=0.87,
               label='Kich ban 2 - XGBoost  (theta=0.3892)   Tang Truong',
               edgecolor='white', linewidth=1.5, zorder=3)

# --- Gia tri tren dau cot (dam va ro rang) ---
ax.bar_label(bars1, fmt='%.1f M', padding=6, fontsize=11,
             fontweight='bold', color='#1A6FAD')
ax.bar_label(bars2, fmt='%.1f M', padding=6, fontsize=11,
             fontweight='bold', color='#0D47A1')

# --- Mu ten 1: FN Cost - giam tu KB1 (45.0M) xuong KB2 (29.7M) ---
# Dat diem bat dau / ket thuc MUI TEN cao hon bar_label it nhat 7 don vi
fn_start_y = kb1_vals[0] + 7    # 52.0  (bar_label KB1_FN o ~46.3)
fn_end_y   = kb2_vals[0] + 7    # 36.7  (bar_label KB2_FN o ~31.0)
ax.annotate('',
    xy    =(x[0] + w/2, fn_end_y),
    xytext=(x[0] - w/2, fn_start_y),
    arrowprops=dict(arrowstyle='->', color='#E53935', lw=2.0,
                    connectionstyle='arc3,rad=-0.28'))
# Text box dat tren DIEM BAT DAU mu ten (52.0) + 11 = 63.0 -> khong de len so lieu
ax.text(x[0], fn_start_y + 11,
        'Giam 15.3M nguy co mat khach',
        ha='center', va='center', fontsize=9.5,
        color='#C62828', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.38', facecolor='#FFF3F3',
                  edgecolor='#E53935', alpha=0.92, linewidth=0.9))

# --- Mu ten 2: FP Cost - tang tu KB1 (55.9M) len KB2 (64.8M) ---
fp_start_y = kb1_vals[1] + 7    # 62.9  (bar_label KB1_FP o ~57.2)
fp_end_y   = kb2_vals[1] + 7    # 71.8  (bar_label KB2_FP o ~66.1)
ax.annotate('',
    xy    =(x[1] + w/2, fp_end_y),
    xytext=(x[1] - w/2, fp_start_y),
    arrowprops=dict(arrowstyle='->', color='#FB8C00', lw=2.0,
                    connectionstyle='arc3,rad=-0.28'))
# Text box dat tren DIEM KET THUC mu ten (71.8) + 9 = 80.8 -> khong de len so lieu
ax.text(x[1], fp_end_y + 9,
        'Ton them 8.9M tien voucher',
        ha='center', va='center', fontsize=9.5,
        color='#E65100', fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.38', facecolor='#FFF8F0',
                  edgecolor='#FB8C00', alpha=0.92, linewidth=0.9))

# --- Hop thong tin tong chi phi (goc tren phai) ---
total_text = ('Tong Chi Phi Thiet Hai:\n'
              '  KB1  LightGBM : 100.9 Trieu VND\n'
              '  KB2  XGBoost  :  94.5 Trieu VND')
ax.text(0.985, 0.985, total_text,
        transform=ax.transAxes, fontsize=10,
        va='top', ha='right',
        bbox=dict(boxstyle='round,pad=0.6', facecolor='#EBF5FF',
                  edgecolor='#1565C0', alpha=0.93, linewidth=1.2))

# --- Dinh dang dashboard ---
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=13, fontweight='bold')
ax.set_ylabel('Chi Phi Uoc Tinh (Trieu VND)', fontsize=12)
ax.set_ylim(0, 108)
ax.set_title(
    'SO SÁNH ĐÁNH ĐỔI KINH DOANH - 2 KỊCH BẢN CHIẾN LƯỢC\n'
    'Kich ban 1: VẬN HÀNH BỀN VỮNG    Kich ban 2: TĂNG TRƯỞNG THỊ PHẦN',
    fontsize=13, fontweight='bold', pad=16)
ax.legend(loc='upper left', fontsize=10, framealpha=0.88,
          edgecolor='#AACCEE', facecolor='#F4F9FF')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_alpha(0.3)
ax.spines['bottom'].set_alpha(0.3)
ax.yaxis.grid(True, linestyle='--', alpha=0.35, color='#AABBCC', zorder=0)
ax.set_axisbelow(True)
ax.tick_params(axis='both', length=0)

plt.tight_layout()
plt.show()

print('  Tom tat danh doi:')
print(f'  {"":5} {"FN Cost (M)":>12} {"FP Cost (M)":>12} {"Total (M)":>11}')
print(f'  {"─"*46}')
print(f'  {"KB1 - LightGBM theta=0.5041":<28} {kb1_vals[0]:>10.1f} {kb1_vals[1]:>12.1f} {sum(kb1_vals):>11.1f}')
print(f'  {"KB2 - XGBoost  theta=0.3892":<28} {kb2_vals[0]:>10.1f} {kb2_vals[1]:>12.1f} {sum(kb2_vals):>11.1f}')
print(f'  {"Delta (KB2 - KB1)":<28} {kb2_vals[0]-kb1_vals[0]:>+10.1f} {kb2_vals[1]-kb1_vals[1]:>+12.1f} {sum(kb2_vals)-sum(kb1_vals):>+11.1f}')
